# 5.18 — Selecting Numerical and Categorical Features

Before modeling, explicitly define:

- **Target** (what you predict)
- **Numerical features** (quantities where arithmetic makes sense)
- **Categorical features** (labels / groups)
- **Excluded columns** (IDs, leakage, post-outcome fields)

This repo keeps the feature lists in `src/config.py` and uses them in preprocessing via `ColumnTransformer`.

In [ ]:
import pandas as pd

from src.data_loader import load_csv
from src.config import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    EXCLUDED_COLUMNS,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
    TARGET_SOURCE_COLUMN,
)
from src.preprocessing import separate_features_and_target, split_data, fit_preprocessor, transform_features

## 1) Load data and inspect the schema

Type selection is not just about `dtype`. Always inspect:

- `df.dtypes`
- missingness
- unique counts (`nunique`) for categorical candidates
- numeric distributions (`describe`)

In [ ]:
df = load_csv("data/raw/source_demo_crops_20260321.csv")

print("shape:", df.shape)
print("dtypes:\n", df.dtypes)
print("missing values:\n", df.isna().sum())
print("nunique:\n", df.nunique(dropna=False))

df.head()

## 2) Define the target first (and remove leakage sources)

For the demo dataset, we derive a binary target from `yield_kg` (high vs low).

Key discipline:

- Define target first
- Remove target + post-outcome columns from feature consideration

In [ ]:
working = df.copy()
working[TARGET_COLUMN] = (working[TARGET_SOURCE_COLUMN] >= working[TARGET_SOURCE_COLUMN].median()).astype(int)

print("Target distribution (normalized):")
print(working[TARGET_COLUMN].value_counts(normalize=True))

## 3) Use explicit feature type lists (the project source of truth)

These lists control preprocessing:

- `NUMERICAL_FEATURES` → scaled with `StandardScaler`
- `CATEGORICAL_FEATURES` → encoded with `OneHotEncoder`
- `EXCLUDED_COLUMNS` → never used as inputs

They are designed to be reviewable in code review and reproducible across runs.

In [ ]:
print("NUMERICAL_FEATURES:", NUMERICAL_FEATURES)
print("CATEGORICAL_FEATURES:", CATEGORICAL_FEATURES)
print("EXCLUDED_COLUMNS:", EXCLUDED_COLUMNS)
print("ALL_FEATURES:", ALL_FEATURES)

## 4) Validate feature/target separation (leakage guardrail)

This step fails fast if:

- target is accidentally included as a feature
- excluded columns are mistakenly included
- any configured feature column is missing

In [ ]:
X, y = separate_features_and_target(
    working,
    target_column=TARGET_COLUMN,
    feature_columns=[c for c in ALL_FEATURES if c in working.columns],
    excluded_columns=EXCLUDED_COLUMNS,
    verbose=True,
)

X.head()

## 5) See how feature types affect preprocessing output

`fit_preprocessor(...)` learns transformations **only from `X_train`**.

This is where your numeric vs categorical selection becomes concrete:

- Numeric features become standardized columns
- Categorical features become one-hot columns

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

bundle = fit_preprocessor(
    X_train,
    numeric_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

X_train_t = transform_features(X_train, bundle)
X_test_t = transform_features(X_test, bundle)

print("X_train transformed shape:", X_train_t.shape)
print("X_test transformed shape:", X_test_t.shape)
print("Transformed columns:")
print(list(X_train_t.columns))

## 6) Practical checklist before modeling

- Target defined and removed from features
- Numerical and categorical lists explicitly defined
- No overlap between numeric and categorical lists
- No excluded columns in features
- All features exist in the dataset
- Categorical columns checked for:
  - unexpected levels
  - high cardinality (`nunique` is very large)
- Every feature passes the **prediction-moment test** (available at prediction time)

In [ ]:
# Quick diagnostics to review categorical health
cat_cardinality = {col: int(df[col].nunique(dropna=False)) for col in CATEGORICAL_FEATURES if col in df.columns}
cat_cardinality